In [ ]:
# ---- which sweeps "Run All" runs: True = run, False = skip -------------------------------------
# every sweep cell below checks its flag; a staged sweep runs all its stages in order, each on top of the best values
# of the stages before. Run time per run on the RTX 5080 where measured
sweep_bias_model = False                        # 24 runs, seconds each (closed form)
sweep_svd = False                               # 4 runs
sweep_svdpp = False                             # 4 runs
sweep_item_knn = False                          # 6 runs (CPU)
sweep_zero_model = False                        # 9 runs (log loss of P(0))
sweep_time_bias_model = False                   # 9 runs, closed form
sweep_time_svdpp = False                        # 3 runs, about as long as SVD++
sweep_time_item_knn = False                     # 9 runs (CPU)
sweep_time_zero_model = False                   # 3 runs (log loss of P(0))
sweep_time_factorization_machine = True         # 15 runs in 3 stages, ~1.5 min (k = 120) ... ~10 min (k = 500) each

# Task 3 hyperparameter sweeps

**Run All:** set the flags in the first cell, then run the whole notebook. Every sweep cell only runs if its flag is `True`.
A staged sweep (time-aware factorization machine) runs all its stages in order; every stage passes the best values of the
stages before on as constants (`best_values_of`), so it does not depend on copying values into `lib/config.py` in between.
Run a single staged cell by hand only after the stages before it (it needs their `..._best_values`).
A sweep that fails stops Run All there; its finished runs are in its output file.

`run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 3 section)
- **nested settings** as `"outer.inner"`, e.g. `"temporal_biases.drift_regularization"`

Every run trains on the training split and is scored on the validation split: **validation MSE (lower is better); the zero models are scored by the log loss of P(rating = 0)**.
No refit on all data, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task3/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py`), the swept and constant values and the score of every run.
For a staged sweep: copy `best.parameters` of its last stage (before the noise check) - it contains the earlier stages' best values.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** `rerun_finalists_on_other_splits=True` re-trains the best `n_finalists` (default 3) combinations on the validation splits with seeds 1 and 2; the winner is then the best mean over the three splits.
`show_training_log=True` prints the training's own log lines of every run.

**Linked settings** (see `lib/config.py`):
- `bias_model` sweeps `TASK3_SHARED_BASELINE_BIASES`, the baseline the item kNN and the time-decayed kNN use too
  -> after changing it, retrain all three (+ the blend). Do not sweep the kNNs' `baseline_biases` on their own.
- `TASK3_TEMPORAL_BIASES` is used by the time-aware bias model and by timeSVD++ (its starting biases)
  -> sweep it via `time_bias_model` (closed form, fast); a change affects both models.
- the zero model and the time zero model enter the blend as a pair (weights +1.8 / -1.8) -> keep their prior strengths equal.

In [ ]:
%load_ext autoreload
# --full: reload whole modules. The default only re-runs the changed lines of a module, so a setting computed from an
# edited one (e.g. TASK1_BLENDED_MODELS from TASK1_BLEND_MODEL_FLAGS in lib/config.py) would keep its old value
%autoreload 2 --full

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import best_values_of, print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## Bias model = shared baseline (closed form, seconds per run)

In [ ]:
if sweep_bias_model:
    bias_model_sweep = run_grid_search("task3", "bias_model", {
        "user_bias_regularization": [1.0, 2.0, 3.0, 5.0],
        "item_bias_regularization": [3.0, 5.0, 8.0],
        "n_iterations": [10, 50],
    })

## SVD / SVD++ (GPU)

In [ ]:
if sweep_svd:
    svd_sweep = run_grid_search("task3", "svd", {
        "user_bias_regularization": [1.0, 3.0],
        "item_bias_regularization": [3.0, 5.0],
    })

In [ ]:
if sweep_svdpp:
    svdpp_sweep = run_grid_search("task3", "svdpp", {
        "user_bias_regularization": [1.0, 3.0],
        "item_bias_regularization": [3.0, 5.0],
    }, rerun_finalists_on_other_splits=False, n_finalists=2)

## Item kNN (CPU; baseline = the shared bias model)

In [ ]:
if sweep_item_knn:
    item_knn_sweep = run_grid_search("task3", "item_knn", {
        "n_neighbors": [50, 100, 200],
        "prediction_shrinkage": [0.3, 1.0],
    })

## Zero model (log loss of P(0))

In [ ]:
if sweep_zero_model:
    zero_model_sweep = run_grid_search("task3", "zero_model", {
        "user_prior_strength": [2.0, 5.0, 10.0],
        "item_prior_strength": [10.0, 20.0, 40.0],
    })

## Time-aware bias model (closed form; sweeps the temporal biases shared with timeSVD++)

In [ ]:
if sweep_time_bias_model:
    time_bias_model_sweep = run_grid_search("task3", "time_bias_model", {
        "temporal_biases.time_bias_regularization": [25.0, 50.0, 100.0],
        "temporal_biases.drift_regularization": [1000.0, 3000.0, 10000.0],
    })

## timeSVD++ (GPU, about as long as SVD++)

In [ ]:
if sweep_time_svdpp:
    time_svdpp_sweep = run_grid_search("task3", "time_svdpp", {
        "factor_regularization": [4.0, 8.0, 16.0],
    })

## Time-decayed item kNN (CPU)

In [ ]:
if sweep_time_item_knn:
    time_item_knn_sweep = run_grid_search("task3", "time_item_knn", {
        "time_decay_days": [90.0, 365.0, 1000.0],
        "time_decay_floor": [0.3, 0.5, 0.7],
    })

## Time zero model (log loss of P(0))

In [ ]:
if sweep_time_zero_model:
    time_zero_model_sweep = run_grid_search("task3", "time_zero_model", {
        "window_prior_strength": [1.0, 2.0, 5.0],
    })

## Time-aware factorization machine (GPU)
The stages of the Task 1 factorization machine on top of `TASK3_TIME_FACTORIZATION_MACHINE` in `lib/config.py`; every stage
starts from the best values of the stages before.

1. **Calendar fields** - done 14 Nov: all four 0.89674, weekday + month + year 0.89555, **month + year 0.89413**, year 0.89543
   -> month + year, now in `lib/config.py`. (`hour_of_day` carries little: every timestamp lies between 05:00 and 09:59 UTC and that
   window moves with US daylight saving time.)
2. **Optimisation**: `learning_rate` x `ratings_per_batch` (Task 1: 5e-4 / 256 was best, the config still has 2.5e-4 / 512)
3. **Capacity and regularisation**: `n_factors` x `factor_regularization`
4. **Bias lambda** (one lambda for b_u and b_i)

Not swept (as in Task 1): `n_activity_buckets` (within the noise there), the seed check (noise of one run: ~+-0.0004).

In [ ]:
if sweep_time_factorization_machine:
    # stage 2 - optimisation: learning rate x batch size (6 runs; batch 256 takes ~2x as long as 512)
    time_factorization_machine_optimisation_sweep = run_grid_search("task3", "time_factorization_machine", {
        "learning_rate": [2.5e-4, 5e-4, 1e-3],
        "ratings_per_batch": [256, 512],
    })
    time_factorization_machine_best_values = best_values_of(time_factorization_machine_optimisation_sweep)

In [ ]:
if sweep_time_factorization_machine:
    # stage 3 - capacity x regularisation (6 runs; time ~ proportional to k)
    time_factorization_machine_capacity_sweep = run_grid_search("task3", "time_factorization_machine", {
        **time_factorization_machine_best_values,
        "n_factors": [120, 250, 500],
        "factor_regularization": [8.0, 16.0],
    })
    time_factorization_machine_best_values |= best_values_of(time_factorization_machine_capacity_sweep)

In [ ]:
if sweep_time_factorization_machine:
    # stage 4 - one lambda for b_u and b_i (also the closed-form starting biases) (3 runs)
    time_factorization_machine_bias_sweep = run_grid_search("task3", "time_factorization_machine", {
        **time_factorization_machine_best_values,
        "bias_regularization": [1.0, 3.0, 5.0],
    })

## Overview

In [ ]:
print_sweep_overview("task3")